# Notebook 10 — Temporal Fusion Transformer (TFT)

## Purpose

TFT is the most complex forecasting model in the project.

It combines:

- recurrent sequence processing;
- attention;
- variable selection;
- static and time-varying information.

This notebook is deliberately kept in **quick mode** first. TFT should be treated as optional if time or computation becomes a problem.

In [1]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# Find the project root whether the notebook is launched from:
#   project/
# or:
#   project/notebooks/
CURRENT = Path.cwd().resolve()

if (CURRENT / "data").exists():
    PROJECT_ROOT = CURRENT
elif CURRENT.name == "notebooks" and (CURRENT.parent / "data").exists():
    PROJECT_ROOT = CURRENT.parent
else:
    possible_roots = [CURRENT, *CURRENT.parents]
    matches = [p for p in possible_roots if (p / "data").exists() and (p / "notebooks").exists()]
    if not matches:
        raise FileNotFoundError(
            "Could not find the project root. Open the sp500-forecasting-dissertation "
            "folder in VS Code, then run this notebook again."
        )
    PROJECT_ROOT = matches[0]

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
REPORT_TABLES = PROJECT_ROOT / "reports" / "tables"
REPORT_FIGURES = PROJECT_ROOT / "reports" / "figures"
MODEL_DIR = PROJECT_ROOT / "reports" / "models"

for folder in [DATA_PROCESSED, REPORT_TABLES, REPORT_FIGURES, MODEL_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Python:", sys.executable)
print("Python version:", sys.version.split()[0])

Project root: <project_root>
Python: /opt/anaconda3/envs/dissertation/bin/python
Python version: 3.11.15


In [2]:
def find_crsp_panel(processed_folder: Path) -> Path:
    """Find the best available corrected CRSP parquet file."""
    preferred_names = [
        "crsp_sp500_daily_corrected_2010_2024.parquet",
        "crsp_sp500_daily_corrected.parquet",
        "crsp_daily_corrected_2010_2024.parquet",
        "crsp_daily_2010_2024.parquet",
    ]

    for name in preferred_names:
        path = processed_folder / name
        if path.exists():
            return path

    candidates = sorted(
        processed_folder.glob("*.parquet"),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )

    if not candidates:
        raise FileNotFoundError(
            f"No parquet file was found in {processed_folder}. "
            "Run and save the corrected CRSP data notebook first."
        )

    print("No preferred filename found. Using the newest parquet file:")
    return candidates[0]


PANEL_PATH = find_crsp_panel(DATA_PROCESSED)
panel = pd.read_parquet(PANEL_PATH).copy()
panel.columns = [str(c).strip().lower() for c in panel.columns]

# Accept the possible names used in the earlier audit notebook.
return_candidates = ["ret", "combined_return", "ret_combined"]
return_source = next((c for c in return_candidates if c in panel.columns), None)

required_base = {"permno", "date"}
missing_base = required_base.difference(panel.columns)

if missing_base:
    raise ValueError(f"Missing required columns: {sorted(missing_base)}")

if return_source is None:
    raise ValueError(
        "Could not find a return column. Expected one of: "
        f"{return_candidates}. Found: {panel.columns.tolist()}"
    )

panel["date"] = pd.to_datetime(panel["date"])
panel["permno"] = pd.to_numeric(panel["permno"], errors="coerce").astype("Int64")
panel[return_source] = pd.to_numeric(panel[return_source], errors="coerce")

if "mktcap" in panel.columns:
    panel["mktcap"] = pd.to_numeric(panel["mktcap"], errors="coerce")

panel = (
    panel.dropna(subset=["permno", "date"])
         .sort_values(["date", "permno"])
         .reset_index(drop=True)
)

# IMPORTANT DECISION:
# "simple" keeps CRSP total returns, including a possible -100% delisting return.
# "log" uses log(1 + return), but an exact -100% return cannot be logged.
RETURN_MODE = "simple"

if RETURN_MODE == "simple":
    panel["model_return"] = panel[return_source]
elif RETURN_MODE == "log":
    impossible_for_log = panel[return_source] <= -1
    print("Rows not usable as log-returns:", int(impossible_for_log.sum()))
    panel["model_return"] = np.where(
        panel[return_source] > -1,
        np.log1p(panel[return_source]),
        np.nan,
    )
else:
    raise ValueError("RETURN_MODE must be 'simple' or 'log'.")

print("Loaded:", PANEL_PATH)
print("Rows:", len(panel))
print("Dates:", panel["date"].min(), "to", panel["date"].max())
print("Unique PERMNOs:", panel["permno"].nunique())
print("Return source:", return_source)
print("Return mode:", RETURN_MODE)
print("Missing modelling returns:", panel["model_return"].isna().sum())

Loaded: <project_root>/data/processed/crsp_sp500_daily_corrected_2010_2024.parquet
Rows: 1711517
Dates: 2010-01-04 00:00:00 to 2024-12-31 00:00:00
Unique PERMNOs: 740
Return source: ret
Return mode: simple
Missing modelling returns: 60


In [3]:
def latest_cross_section_before(data: pd.DataFrame, as_of_date) -> pd.DataFrame:
    """
    Return the cross-section on the latest trading date on or before as_of_date.

    We deliberately do NOT take each stock's individually latest historical row.
    Doing that could accidentally include a company that left the S&P 500 years ago.
    Using one common market date keeps only securities present in the corrected panel
    on that date.
    """
    as_of_date = pd.Timestamp(as_of_date)

    eligible_dates = data.loc[
        data["date"] <= as_of_date,
        "date"
    ]

    if eligible_dates.empty:
        raise ValueError(f"No data available on or before {as_of_date.date()}.")

    market_date = eligible_dates.max()

    cross_section = data.loc[
        data["date"] == market_date
    ].copy()

    if cross_section.empty:
        raise ValueError(f"No cross-section found for {market_date.date()}.")

    return cross_section


def select_top_n_by_lagged_market_cap(
    data: pd.DataFrame,
    as_of_date,
    n_stocks: int,
) -> list[int]:
    """
    Select stocks using market capitalisation from one common historical date.
    """
    if "mktcap" not in data.columns:
        raise ValueError("The panel needs a 'mktcap' column for top-N selection.")

    cross_section = latest_cross_section_before(data, as_of_date)
    cross_section = cross_section.dropna(subset=["mktcap"])
    cross_section = cross_section[cross_section["mktcap"] > 0]

    selected = (
        cross_section.nlargest(n_stocks, "mktcap")["permno"]
                     .astype(int)
                     .tolist()
    )

    if len(selected) < n_stocks:
        print(f"Warning: requested {n_stocks} stocks, but selected {len(selected)}.")

    return selected

## Install the extra libraries

Run once in the VS Code terminal:

```bash
python -m pip install "pytorch-forecasting>=1.4,<2" "lightning>=2.2" torch scikit-learn
```

After installation, restart the notebook kernel.

In [4]:
try:
    import torch
    import lightning.pytorch as pl
    from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint

    from pytorch_forecasting import (
        TimeSeriesDataSet,
        TemporalFusionTransformer,
    )
    from pytorch_forecasting.data import GroupNormalizer
    from pytorch_forecasting.metrics import RMSE

except ImportError as error:
    raise ImportError(
        "TFT packages are missing. Install pytorch-forecasting and lightning "
        "using the command in the previous markdown cell, then restart the kernel."
    ) from error

print("PyTorch:", torch.__version__)
print("Lightning:", pl.__version__)

PyTorch: 2.12.1
Lightning: 2.6.5


In [5]:
QUICK_MODE = True

TRAIN_END = pd.Timestamp("2018-12-31")
VALIDATION_END = pd.Timestamp("2019-12-31")

if QUICK_MODE:
    N_STOCKS = 10
    ENCODER_LENGTH = 60
    MAX_EPOCHS = 5
    BATCH_SIZE = 128
else:
    N_STOCKS = 100
    ENCODER_LENGTH = 252
    MAX_EPOCHS = 30
    BATCH_SIZE = 256

PREDICTION_LENGTH = 1

print({
    "n_stocks": N_STOCKS,
    "encoder_length": ENCODER_LENGTH,
    "epochs": MAX_EPOCHS,
})

{'n_stocks': 10, 'encoder_length': 60, 'epochs': 5}


## Prepare long-format panel data

TFT needs:

- one row per stock-date;
- a group identifier (`permno_string`);
- a continuous time index shared by all stocks;
- the return target;
- optional calendar features.

Only market-cap information available by the end of 2018 is used to choose the demonstration universe.

In [6]:
selected_permnos = select_top_n_by_lagged_market_cap(
    panel,
    as_of_date=TRAIN_END,
    n_stocks=N_STOCKS,
)

tft_data = panel[
    panel["permno"].isin(selected_permnos)
][["date", "permno", "model_return"]].dropna().copy()

all_dates = (
    panel["date"].drop_duplicates().sort_values().reset_index(drop=True)
)

date_to_time_index = {
    date: index
    for index, date in enumerate(all_dates)
}

tft_data["time_idx"] = tft_data["date"].map(date_to_time_index).astype(int)
tft_data["permno_string"] = tft_data["permno"].astype(str)
tft_data["target"] = tft_data["model_return"].astype(float)

tft_data["day_of_week"] = tft_data["date"].dt.dayofweek.astype(str)
tft_data["month"] = tft_data["date"].dt.month.astype(str)

train_cutoff = max(
    index for date, index in date_to_time_index.items()
    if date <= TRAIN_END
)

validation_cutoff = max(
    index for date, index in date_to_time_index.items()
    if date <= VALIDATION_END
)

print("Rows:", len(tft_data))
print("Groups:", tft_data["permno_string"].nunique())
print("Train cutoff:", train_cutoff)
print("Validation cutoff:", validation_cutoff)

Rows: 35641
Groups: 10
Train cutoff: 2263
Validation cutoff: 2515


## Build the TimeSeriesDataSet

Key idea:

- the encoder sees past returns;
- the decoder predicts the next return;
- group normalisation is learned by stock;
- missing calendar steps are allowed because stocks may have occasional missing observations.

In [7]:
training_data = tft_data[tft_data["time_idx"] <= train_cutoff].copy()

training_dataset = TimeSeriesDataSet(
    training_data,
    time_idx="time_idx",
    target="target",
    group_ids=["permno_string"],

    min_encoder_length=ENCODER_LENGTH,
    max_encoder_length=ENCODER_LENGTH,
    min_prediction_length=PREDICTION_LENGTH,
    max_prediction_length=PREDICTION_LENGTH,

    static_categoricals=["permno_string"],
    time_varying_known_categoricals=["day_of_week", "month"],
    time_varying_known_reals=["time_idx"],
    time_varying_unknown_reals=["target"],

    target_normalizer=GroupNormalizer(
        groups=["permno_string"],
        transformation=None,
    ),

    add_relative_time_idx=True,
    add_target_scales=True,
    add_encoder_length=True,
    allow_missing_timesteps=True,
)

validation_data = tft_data[
    tft_data["time_idx"] <= validation_cutoff
].copy()

validation_dataset = TimeSeriesDataSet.from_dataset(
    training_dataset,
    validation_data,
    min_prediction_idx=train_cutoff + 1,
    stop_randomization=True,
)

test_dataset = TimeSeriesDataSet.from_dataset(
    training_dataset,
    tft_data,
    min_prediction_idx=validation_cutoff + 1,
    stop_randomization=True,
)

train_loader = training_dataset.to_dataloader(
    train=True,
    batch_size=BATCH_SIZE,
    num_workers=0,
)

validation_loader = validation_dataset.to_dataloader(
    train=False,
    batch_size=BATCH_SIZE,
    num_workers=0,
)

test_loader = test_dataset.to_dataloader(
    train=False,
    batch_size=BATCH_SIZE,
    num_workers=0,
)

print("Training samples:", len(training_dataset))
print("Validation samples:", len(validation_dataset))
print("Test samples:", len(test_dataset))

Training samples: 19941
Validation samples: 2520
Test samples: 12580


## Define and train TFT

We use RMSE as the training loss so the model produces one point forecast.

The model is intentionally small in quick mode.

In [8]:
pl.seed_everything(42, workers=True)

early_stopping = EarlyStopping(
    monitor="val_loss",
    min_delta=1e-5,
    patience=3,
    mode="min",
)

checkpoint_callback = ModelCheckpoint(
    dirpath=MODEL_DIR / "tft_checkpoints",
    filename="tft-{epoch:02d}-{val_loss:.6f}",
    monitor="val_loss",
    mode="min",
    save_top_k=1,
)

trainer = pl.Trainer(
    max_epochs=MAX_EPOCHS,
    accelerator="auto",
    devices=1,
    gradient_clip_val=0.1,
    callbacks=[early_stopping, checkpoint_callback],
    logger=False,
    enable_model_summary=True,
    log_every_n_steps=10,
)

tft = TemporalFusionTransformer.from_dataset(
    training_dataset,
    learning_rate=1e-3,
    hidden_size=16 if QUICK_MODE else 32,
    attention_head_size=2 if QUICK_MODE else 4,
    dropout=0.2,
    hidden_continuous_size=8,
    output_size=1,
    loss=RMSE(),
    reduce_on_plateau_patience=2,
    log_interval=-1,
)

print("Number of model parameters:", tft.size())

Seed set to 42
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Number of model parameters: 19291


In [9]:
trainer.fit(
    tft,
    train_dataloaders=train_loader,
    val_dataloaders=validation_loader,
)

┏━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃    ┃ Name                               ┃ Type                            ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0  │ loss                               │ RMSE                            │      0 │ train │     0 │
│ 1  │ logging_metrics                    │ ModuleList                      │      0 │ train │     0 │
│ 2  │ input_embeddings                   │ MultiEmbedding                  │    152 │ train │     0 │
│ 3  │ prescalers                         │ ModuleDict                      │     96 │ train │     0 │
│ 4  │ static_variable_selection          │ VariableSelectionNetwork        │  1.8 K │ train │     0 │
│ 5  │ encoder_variable_selection         │ VariableSelectionNetwork        │  2.0 K │ train │     0 │
│ 6  │ decoder_variable_selection         │ VariableSelectionNetwork        │  1.4 K │ train │     0 │
│ 7  │ static_context_variable_selection  │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 8  │ static_context_initial_hidden_lstm │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 9  │ static_context_initial_cell_lstm   │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 10 │ static_context_enrichment          │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 11 │ lstm_encoder                       │ LSTM                            │  2.2 K │ train │     0 │
│ 12 │ lstm_decoder                       │ LSTM                            │  2.2 K │ train │     0 │
│ 13 │ post_lstm_gate_encoder             │ GatedLinearUnit                 │    544 │ train │     0 │
│ 14 │ post_lstm_add_norm_encoder         │ AddNorm                         │     32 │ train │     0 │
│ 15 │ static_enrichment                  │ GatedResidualNetwork            │  1.4 K │ train │     0 │
│ 16 │ multihead_attn                     │ InterpretableMultiHeadAttention │    808 │ train │     0 │
│ 17 │ post_attn_gate_norm                │ GateAddNorm                     │    576 │ train │     0 │
│ 18 │ pos_wise_ff                        │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 19 │ pre_output_gate_norm               │ GateAddNorm                     │    576 │ train │     0 │
│ 20 │ output_layer                       │ Linear                          │     17 │ train │     0 │
└────┴────────────────────────────────────┴─────────────────────────────────┴────────┴───────┴───────┘

Trainable params: 19.3 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 19.3 K                                                                                               
Total estimated model params size (MB): 0.077                                                                      
Modules in train mode: 298                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

`Trainer.fit` stopped: `max_epochs=5` reached.


## Generate test predictions

`return_index=True` asks PyTorch Forecasting to return the stock and time index associated with each prediction.

The exact returned object can vary slightly across package versions, so the next cell prints its structure before saving.

In [10]:
best_checkpoint = checkpoint_callback.best_model_path

if best_checkpoint:
    best_tft = TemporalFusionTransformer.load_from_checkpoint(best_checkpoint)
else:
    best_tft = tft

prediction_output = best_tft.predict(
    test_loader,
    mode="prediction",
    return_index=True,
    return_y=True,
    trainer_kwargs={
        "accelerator": "auto",
        "devices": 1,
        "logger": False,
    },
)

print("Prediction object type:", type(prediction_output))
print("Available fields:", getattr(prediction_output, "_fields", "not a named tuple"))

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


Prediction object type: <class 'pytorch_forecasting.models.base._base_model.Prediction'>
Available fields: ('output', 'x', 'index', 'decoder_lengths', 'y')


In [11]:
# Extract predictions in a version-tolerant way.

if hasattr(prediction_output, "prediction"):
    prediction_tensor = prediction_output.prediction
elif hasattr(prediction_output, "output"):
    prediction_tensor = prediction_output.output
else:
    prediction_tensor = prediction_output[0]

prediction_values = (
    prediction_tensor.detach().cpu().numpy().reshape(-1)
)

index_frame = (
    prediction_output.index.copy()
    if hasattr(prediction_output, "index")
    else prediction_output[3].copy()
)

# y is commonly a tuple: (target, weight)
y_object = prediction_output.y if hasattr(prediction_output, "y") else prediction_output[2]
actual_tensor = y_object[0] if isinstance(y_object, (tuple, list)) else y_object

actual_values = actual_tensor.detach().cpu().numpy().reshape(-1)

tft_predictions = index_frame.reset_index(drop=True).copy()
tft_predictions["model"] = "tft"
tft_predictions["forecast"] = prediction_values
tft_predictions["actual"] = actual_values

# Convert the first prediction time index back to a date.
time_to_date = {value: key for key, value in date_to_time_index.items()}
tft_predictions["date"] = tft_predictions["time_idx"].map(time_to_date)

tft_predictions.head()

,time_idx,permno_string,model,forecast,actual,date
0,2516,10107,tft,0.000300,0.018516,2020-01-02
1,2517,10107,tft,-0.000613,-0.012452,2020-01-03
2,2518,10107,tft,-0.000315,0.002585,2020-01-06
3,2519,10107,tft,0.000464,-0.009118,2020-01-07
4,2520,10107,tft,0.000675,0.015928,2020-01-08


In [12]:
tft_predictions.to_parquet(
    DATA_PROCESSED / "tft_predictions.parquet",
    index=False,
)

print("Saved TFT predictions.")
print("Best checkpoint:", best_checkpoint)

Saved TFT predictions.
Best checkpoint: <project_root>/reports/models/tft_checkpoints/tft-epoch=01-val_loss=0.013421.ckpt


## Important limitation

This notebook is a first TFT implementation, not yet the final monthly walk-forward TFT experiment.

TFT is computationally expensive. Complete the LSTM and GRU comparison first. Keep TFT only if it remains achievable within the dissertation schedule.

## What to say in the meeting

> TFT is included because it can combine sequence memory, variable selection and attention. I am treating it as the most complex model and therefore the first candidate to reduce if computation becomes a problem. It must be judged under the same time splits and metrics as the simpler models.